# m06_l01_unit

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Unit

In [ ]:
from shopassist.agent import StepLimitReached, run_agent
from shopassist.messages import Message, ToolCall
from shopassist.models import ScriptedModel
from shopassist.tools import GET_ORDER

ASK = Message("assistant", tool_calls=(
    ToolCall("c1", "get_order", {"order_id": "1042"}),))
DONE = Message("assistant", "It is packed.")


def test_tool_result_reaches_the_model():
    model = ScriptedModel([ASK, DONE])
    answer, msgs = run_agent(model, [GET_ORDER], "s", "#1042?")
    assert answer == "It is packed." and '"packed"' in msgs[2].text


def test_unknown_tool_is_reported():
    bad = Message("assistant", tool_calls=(ToolCall("c", "nope", {}),))
    model = ScriptedModel([bad, DONE])
    _, msgs = run_agent(model, [GET_ORDER], "s", "x")
    assert msgs[2].is_error


def test_runaway_is_stopped():
    try:
        run_agent(ScriptedModel([ASK]), [GET_ORDER], "s", "x",
                  max_steps=3)
    except StepLimitReached:
        return
    raise AssertionError("the loop did not stop")


for test in (test_tool_result_reaches_the_model,
             test_unknown_tool_is_reported, test_runaway_is_stopped):
    test()
    print("PASS", test.__name__)
